# EDA: End-to-End Supply Chain Analytics
**Project:** E-Commerce Supply Chain & Inventory Optimization  
**Target Employers:** Amazon, Costco, Walmart, Boeing  
**Objective:** Establish the data foundation for Demand Forecasting, Inventory Optimization, and Late Delivery Risk Prediction.  

---

### Phase 1 & 2 Blueprint Alignment
This notebook aligns directly with our technical roadmap:
1. **Data Quality Audit** — Validating the 180k+ DataCo records.
2. **Demand Forecasting (Time Series)** — Order patterns, seasonality, and sales trends.
3. **Product & Inventory Analysis** — High-volume categories and profit margins (ABC analysis foundation).
4. **Geography & Customer Segments** — Omnichannel mapping for Walmart/Costco personas.
5. **Late Delivery Risk** — Establishing our baseline and preventing data leakage for binary classification.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

# Styling for business-ready charts
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.titleweight'] = 'bold'

DATA_PATH = Path('../data/raw/DataCoSupplyChainDataset.csv')
print(f'Loading data from: {DATA_PATH.resolve()}')

df = pd.read_csv(DATA_PATH, encoding='latin-1')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

---
## 1. Data Quality & Preprocessing
To do time-series forecasting (Demand Forecasting), we must properly parse order dates.

In [ ]:
# Parse datetime
df['order_date'] = pd.to_datetime(df['order date (DateOrders)'])
df['order_year_month'] = df['order_date'].dt.to_period('M')

# Missing values summary
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) > 0:
    print("Missing Values:")
    display(pd.DataFrame({'Missing': missing, '%': (missing/len(df)*100).round(2)}))
else:
    print("No missing values detected in core columns.")

---
## 2. Demand Forecasting: Seasonality & Order Patterns
*Key for Amazon/Walmart demand prediction models.*

In [ ]:
# Monthly Sales & Order Volume Trend
monthly_sales = df.groupby('order_year_month')['Sales'].sum()

fig, ax = plt.subplots(figsize=(12, 4))
monthly_sales.plot(kind='line', ax=ax, color='#2E86C1', linewidth=2, marker='o')
ax.set_title('Total Sales over Time (Demand Seasonality)')
ax.set_ylabel('Total Revenue ($)')
ax.set_xlabel('Order Month')
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
plt.tight_layout()
plt.show()

---
## 3. Inventory Optimization Context: ABC Analysis Foundation
*Identifying which departments and product categories drive the most volume and profit, useful for Costco/Amazon warehouse logistics.*

In [ ]:
# Top 10 Departments by Order Volume and Profit
dept_summary = df.groupby('Department Name').agg({
    'Order Id': 'count',
    'Order Item Profit Ratio': 'mean',
    'Sales': 'sum'
}).rename(columns={'Order Id': 'Order Volume', 'Order Item Profit Ratio': 'Avg Profit Ratio'}).sort_values('Order Volume', ascending=False).head(10)

display(dept_summary.style.format({'Avg Profit Ratio': '{:.2%}', 'Sales': '${:,.0f}'}))

---
## 4. Customer Segmentation & Geography
*Understanding the omnichannel scope.*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Segment
segment_counts = df['Customer Segment'].value_counts()
axes[0].pie(segment_counts, labels=segment_counts.index, autopct='%1.1f%%', colors=sns.color_palette('pastel'))
axes[0].set_title('Customer Segments')

# Market
market_sales = df.groupby('Market')['Sales'].sum().sort_values()
market_sales.plot(kind='barh', ax=axes[1], color='#27AE60')
axes[1].set_title('Total Sales by Global Market')
axes[1].xaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))

plt.tight_layout()
plt.show()

---
## 5. Late Delivery Risk: Target Baseline & Leakage Audit
*Preparing the binary classification target. Dropping target leakage features (e.g. actual shipping days) to ensure model validity.*

In [ ]:
target = 'Late_delivery_risk'
class_counts = df[target].value_counts()
late_rate = class_counts.get(1, 0) / len(df) * 100

print(f"Overall Late Delivery Rate: {late_rate:.2f}%")

# Leakage prevention
LEAKAGE_COLUMNS = [
    'Days for shipping (real)',   
    'Delivery Status',            
    'shipping date (DateOrders)'  
]

df_clean = df.drop(columns=[c for c in LEAKAGE_COLUMNS if c in df.columns])

# Top correlated numerical features with Late Delivery Risk
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
correlations = df_clean[numeric_cols].corr()[target].drop(target)
print("\nTop 5 features correlated with Late Delivery Risk:")
print(correlations.abs().sort_values(ascending=False).head(5).round(4))